# How Many Samples Before You Believe It

**Topic:** Monte Carlo estimation &nbsp;·&nbsp; **Difficulty:** Medium

You need P(the sum of 20 service latencies exceeds a deadline). There is no
closed form, so you simulate. The question your manager will actually ask is not
"what is the estimate" but "how sure are you", and that has an answer.

The standard error of a Monte Carlo proportion is `sqrt(p(1-p)/n)`, which falls
like `1/sqrt(n)`. Four times the samples buys you half the error, and no more.

## The data (created in the next cell)

- Each of 20 stages has a Gamma latency with its own shape and scale.
- `deadline = 61.0` milliseconds for the total.
- Use `np.random.default_rng(2024)` so results are reproducible.

## Your task

With **NumPy only**:

1. Simulate `n = 400_000` totals: for each trial sum one Gamma draw per stage,
   using the shapes and scales given.
2. Estimate `p_hat` = fraction of totals exceeding `deadline`.
3. Compute the standard error `sqrt(p(1-p)/n)` and the 95% interval
   `p_hat +- 1.96 * se`.
4. Compute `n_needed`: the smallest `n` making the half-width `1.96*se` fall
   below `0.0005`, using your `p_hat` (round up to an integer).
5. Set `answers` to a dict with `"p_hat"`, `"se"`, `"ci_low"`, `"ci_high"`,
   `"n_needed"`,
   each a plain number (a scalar, not an array).

## Evaluation

Every value is checked to a tolerance of `1e-6 * (1 + |reference|)` against the draw
`np.random.default_rng(2024).gamma(shape=shapes, scale=scales, size=(n, len(shapes)))`,
so use exactly that draw.

In [ ]:
import numpy as np

shapes = np.array([2.0, 3.0, 1.5, 2.5, 4.0, 2.0, 3.5, 1.2, 2.8, 3.2,
                   2.1, 1.8, 2.6, 3.1, 1.4, 2.9, 2.2, 3.6, 1.9, 2.4])
scales = np.array([1.1, 0.8, 1.4, 0.9, 0.7, 1.2, 0.85, 1.6, 1.0, 0.95,
                   1.05, 1.3, 0.92, 0.88, 1.5, 0.98, 1.15, 0.8, 1.25, 1.02])
deadline = 61.0
n = 400_000
print(f"{len(shapes)} stages, mean total = {(shapes * scales).sum():.2f} ms, "
      f"deadline = {deadline} ms")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the parameters created above to simulate the totals, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("p_hat", [], 0.066915, -0.011204114761312212, 1e-06),
    ("se", [], 0.0003950866448483168, -6.615252349312324e-05, 1e-06),
    ("ci_low", [], 0.0661406301760973, -0.011074455815265689, 1e-06),
    ("ci_high", [], 0.06768936982390271, -0.011333773707358734, 1e-06),
    ("n_needed", [], 959438.0, -160646.39405759345, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Draw the whole `(n, 20)` block at once and sum along the stage axis. Looping
over trials is many times slower for no benefit. The sample size needed for a
target half-width `w` inverts the standard-error formula:
`n >= p(1-p) (1.96/w)^2`, which is why chasing another decimal place costs a
hundred times the compute.

```python
import numpy as np

rng = np.random.default_rng(2024)
draws = rng.gamma(shape=shapes, scale=scales, size=(n, len(shapes)))
totals = draws.sum(axis=1)

p_hat = float((totals > deadline).mean())
se = float(np.sqrt(p_hat * (1 - p_hat) / n))

answers = {
    "p_hat": p_hat,
    "se": se,
    "ci_low": p_hat - 1.96 * se,
    "ci_high": p_hat + 1.96 * se,
    "n_needed": int(np.ceil(p_hat * (1 - p_hat) * (1.96 / 0.0005) ** 2)),
}
```